The Big Question
For any macro signal, we want to answer:
Is this signal real, useful, and robust?

I will try to build a very basic signal validation framework here using one example: **Does the US yield curve slope predict future HSI returns?**



Q1: Does the signal have an economic story?

A1: Yes, a steeper yield curve may indicate healthier growth expectations and easier future financial conditions, which can support risk assets like HSI. A flatter or inverted curve may signal slowdown/recession risk or tight monetary conditions, which can hurt equity returns.

In [ ]:
# Q2: Is the coefficient statistically significant?

import pandas as pd
import yfinance as yf
from scipy.stats import ttest_ind
import numpy as np
from scipy import stats

# ------------------------------------------------------------
# Project: Does the US yield curve slope predict future HSI returns?
# ------------------------------------------------------------
# Data:
# - HSI price data from Yahoo Finance
# - US 10Y Treasury yield from FRED
# - US 2Y Treasury yield from FRED
#
# Signal:
# yield_curve_slope = US_10Y_yield - US_2Y_yield
#
# Target:
# future HSI monthly return
# ------------------------------------------------------------

start_date = "2010-01-01"
end_date = None

# ------------------------------------------------------------
# 1. Import HSI data
# ------------------------------------------------------------

hsi_data = yf.download(
    "^HSI",
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

hsi_prices = hsi_data["Close"]

if isinstance(hsi_prices, pd.DataFrame):
    hsi_prices = hsi_prices.iloc[:, 0]

hsi_prices.name = "HSI_price"

# ------------------------------------------------------------
# 2. Import US Treasury yield data from FRED
# ------------------------------------------------------------

dgs10_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
dgs2_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS2"

dgs10 = pd.read_csv(dgs10_url)
dgs2 = pd.read_csv(dgs2_url)

dgs10["observation_date"] = pd.to_datetime(dgs10["observation_date"])
dgs2["observation_date"] = pd.to_datetime(dgs2["observation_date"])

dgs10 = dgs10.set_index("observation_date")
dgs2 = dgs2.set_index("observation_date")

dgs10 = dgs10.rename(columns={"DGS10": "US_10Y_yield"})
dgs2 = dgs2.rename(columns={"DGS2": "US_2Y_yield"})

# Convert missing "." strings to NaN, then convert to numeric
dgs10["US_10Y_yield"] = pd.to_numeric(dgs10["US_10Y_yield"], errors="coerce")
dgs2["US_2Y_yield"] = pd.to_numeric(dgs2["US_2Y_yield"], errors="coerce")

dgs10 = dgs10.loc[start_date:]
dgs2 = dgs2.loc[start_date:]

# ------------------------------------------------------------
# 3. Combine daily data
# ------------------------------------------------------------

daily_data = pd.concat(
    [hsi_prices, dgs10, dgs2],
    axis=1
).sort_index()

# Forward-fill yield data because yields may have missing holidays.
daily_data[["US_10Y_yield", "US_2Y_yield"]] = daily_data[
    ["US_10Y_yield", "US_2Y_yield"]
].ffill()

daily_data = daily_data.dropna()

# ------------------------------------------------------------
# 4. Create yield curve slope
# ------------------------------------------------------------

daily_data["yield_curve_slope"] = (
    daily_data["US_10Y_yield"] - daily_data["US_2Y_yield"]
)

# ------------------------------------------------------------
# 5. Convert to monthly data
# ------------------------------------------------------------
# Month-end signal:
# current month-end yield curve slope -> next month's HSI return
# ------------------------------------------------------------

monthly_data = daily_data.resample("ME").last()

monthly_data["HSI_monthly_return"] = monthly_data["HSI_price"].pct_change()

# Future return target: next month's HSI return
monthly_data["HSI_next_month_return"] = monthly_data["HSI_monthly_return"].shift(-1)

monthly_data = monthly_data.dropna()

# ------------------------------------------------------------
# 6. Final dataset
# ------------------------------------------------------------

final_data = monthly_data[
    [
        "HSI_price",
        "US_10Y_yield",
        "US_2Y_yield",
        "yield_curve_slope",
        "HSI_monthly_return",
        "HSI_next_month_return",
    ]
].copy()

# ------------------------------------------------------------
# Q2: Yield curve regime t-test
# ------------------------------------------------------------

normal_curve_returns = final_data[
    final_data["yield_curve_slope"] > 0
]["HSI_next_month_return"]

inverted_curve_returns = final_data[
    final_data["yield_curve_slope"] <= 0
]["HSI_next_month_return"]

t_stat, p_value = ttest_ind(
    normal_curve_returns,
    inverted_curve_returns,
    equal_var=False,
    nan_policy="omit"
)

print("Yield Curve Regime T-Test")
print(f"Normal curve observations: {len(normal_curve_returns)}")
print(f"Inverted curve observations: {len(inverted_curve_returns)}")
print(f"Normal curve mean next-month HSI return: {normal_curve_returns.mean():.4%}")
print(f"Inverted curve mean next-month HSI return: {inverted_curve_returns.mean():.4%}")
print(f"Difference: {(normal_curve_returns.mean() - inverted_curve_returns.mean()):.4%}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference at the 5% level.")
else:
    print("Result: Not statistically significant at the 5% level.")

# ------------------------------------------------------------
# Q3: Confidence interval for difference in means
# ------------------------------------------------------------
# Difference = normal curve mean - inverted curve mean
#
# This confidence interval asks:
# What range of true regime differences is plausible?
#
# If the interval includes 0, the difference is not statistically clear.
# ------------------------------------------------------------

normal_mean = normal_curve_returns.mean()
inverted_mean = inverted_curve_returns.mean()
difference = normal_mean - inverted_mean

normal_n = len(normal_curve_returns)
inverted_n = len(inverted_curve_returns)

normal_var = normal_curve_returns.var(ddof=1)
inverted_var = inverted_curve_returns.var(ddof=1)

standard_error = np.sqrt(
    normal_var / normal_n +
    inverted_var / inverted_n
)

df_num = (normal_var / normal_n + inverted_var / inverted_n) ** 2

df_den = (
    (normal_var / normal_n) ** 2 / (normal_n - 1)
    + (inverted_var / inverted_n) ** 2 / (inverted_n - 1)
)

degrees_of_freedom = df_num / df_den

confidence_level = 0.95
alpha = 1 - confidence_level

t_critical = stats.t.ppf(
    1 - alpha / 2,
    degrees_of_freedom
)

ci_lower = difference - t_critical * standard_error
ci_upper = difference + t_critical * standard_error

print("\nConfidence Interval For Difference In Means")
print(f"Normal mean: {normal_mean:.4%}")
print(f"Inverted mean: {inverted_mean:.4%}")
print(f"Difference: {difference:.4%}")
print(f"Standard error: {standard_error:.4%}")
print(f"Degrees of freedom: {degrees_of_freedom:.2f}")
print(f"95% CI for difference: [{ci_lower:.4%}, {ci_upper:.4%}]")

if ci_lower <= 0 <= ci_upper:
    print("Interpretation: The confidence interval includes 0, so the regime difference is not statistically clear.")
else:
    print("Interpretation: The confidence interval excludes 0, so the regime difference is statistically clearer.")

Yield Curve Regime T-Test
Normal curve observations: 170
Inverted curve observations: 27
Normal curve mean next-month HSI return: 0.1707%
Inverted curve mean next-month HSI return: 0.6006%
Difference: -0.4299%
T-statistic: -0.2429
P-value: 0.8098
Result: Not statistically significant at the 5% level.

Confidence Interval For Difference In Means
Normal mean: 0.1707%
Inverted mean: 0.6006%
Difference: -0.4299%
Standard error: 1.7696%
Degrees of freedom: 28.72
95% CI for difference: [-4.0507%, 3.1909%]
Interpretation: The confidence interval includes 0, so the regime difference is not statistically clear.


A2: The sample average is higher after inverted-curve months, but there is no strong evidence that yield curve regime meaningfully predicts next-month HSI returns due to high P-Value. The difference between normal-curve and inverted-curve HSI returns is small relative to the noise in the data.

A3: Although inverted yield curve months had a higher average next-month HSI return in the sample, the 95% confidence interval for the difference in means is wide and includes zero. This means the observed difference is not statistically clear and could easily be due to noise. The yield curve regime alone does not provide reliable evidence of different future HSI returns.

In [ ]:
#Q4. How much variance does the signal explain?

import pandas as pd
import yfinance as yf
import statsmodels.api as sm

# ------------------------------------------------------------
# Project:
# Does the US yield curve slope predict future HSI returns?
#
# Model:
# HSI_next_month_return = alpha + beta * yield_curve_slope + error
# ------------------------------------------------------------

start_date = "2010-01-01"
end_date = None

# ------------------------------------------------------------
# 1. Download HSI price data
# ------------------------------------------------------------

hsi_data = yf.download(
    "^HSI",
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

hsi_prices = hsi_data["Close"]

if isinstance(hsi_prices, pd.DataFrame):
    hsi_prices = hsi_prices.iloc[:, 0]

hsi_prices.name = "HSI_price"

# ------------------------------------------------------------
# 2. Download US Treasury yield data from FRED
# ------------------------------------------------------------

dgs10_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
dgs2_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS2"

dgs10 = pd.read_csv(dgs10_url)
dgs2 = pd.read_csv(dgs2_url)

dgs10["observation_date"] = pd.to_datetime(dgs10["observation_date"])
dgs2["observation_date"] = pd.to_datetime(dgs2["observation_date"])

dgs10 = dgs10.set_index("observation_date")
dgs2 = dgs2.set_index("observation_date")

dgs10 = dgs10.rename(columns={"DGS10": "US_10Y_yield"})
dgs2 = dgs2.rename(columns={"DGS2": "US_2Y_yield"})

dgs10["US_10Y_yield"] = pd.to_numeric(
    dgs10["US_10Y_yield"],
    errors="coerce"
)

dgs2["US_2Y_yield"] = pd.to_numeric(
    dgs2["US_2Y_yield"],
    errors="coerce"
)

dgs10 = dgs10.loc[start_date:]
dgs2 = dgs2.loc[start_date:]

# ------------------------------------------------------------
# 3. Combine daily data
# ------------------------------------------------------------

daily_data = pd.concat(
    [hsi_prices, dgs10, dgs2],
    axis=1
).sort_index()

daily_data[["US_10Y_yield", "US_2Y_yield"]] = daily_data[
    ["US_10Y_yield", "US_2Y_yield"]
].ffill()

daily_data = daily_data.dropna()

# ------------------------------------------------------------
# 4. Create yield curve slope
# ------------------------------------------------------------
# yield_curve_slope = 10Y yield - 2Y yield
# ------------------------------------------------------------

daily_data["yield_curve_slope"] = (
    daily_data["US_10Y_yield"] - daily_data["US_2Y_yield"]
)

# ------------------------------------------------------------
# 5. Convert to monthly data
# ------------------------------------------------------------
# Signal:
# month-end yield curve slope
#
# Target:
# next-month HSI return
# ------------------------------------------------------------

monthly_data = daily_data.resample("ME").last()

monthly_data["HSI_monthly_return"] = (
    monthly_data["HSI_price"].pct_change()
)

monthly_data["HSI_next_month_return"] = (
    monthly_data["HSI_monthly_return"].shift(-1)
)

monthly_data = monthly_data.dropna()

# ------------------------------------------------------------
# 6. Final regression dataset
# ------------------------------------------------------------

final_data = monthly_data[
    [
        "yield_curve_slope",
        "HSI_next_month_return",
    ]
].dropna().copy()

print("Final dataset shape:")
print(final_data.shape)

print("\nFirst 5 rows:")
print(final_data.head())

# ------------------------------------------------------------
# 7. Linear regression
# ------------------------------------------------------------
# Y = target variable:
# next-month HSI return
#
# X = predictor/signal:
# current month-end yield curve slope
# ------------------------------------------------------------

Y = final_data["HSI_next_month_return"]
X = final_data["yield_curve_slope"]

# Add intercept / alpha
X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()

print("\nLinear Regression Results")
print(model.summary())

# ------------------------------------------------------------
# 8. Clean key outputs
# ------------------------------------------------------------

alpha = model.params["const"]
beta = model.params["yield_curve_slope"]
beta_p_value = model.pvalues["yield_curve_slope"]
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj

ci_lower = model.conf_int().loc["yield_curve_slope", 0]
ci_upper = model.conf_int().loc["yield_curve_slope", 1]

print("\nKey Signal Validation Outputs")
print(f"Alpha: {alpha:.4%}")
print(f"Beta: {beta:.4%}")
print(f"Beta p-value: {beta_p_value:.4f}")
print(f"R-squared: {r_squared:.4%}")
print(f"Adjusted R-squared: {adj_r_squared:.4%}")
print(f"95% confidence interval for beta: [{ci_lower:.4%}, {ci_upper:.4%}]")

if beta_p_value < 0.05:
    print("Result: Beta is statistically significant at the 5% level.")
else:
    print("Result: Beta is NOT statistically significant at the 5% level.")

if ci_lower <= 0 <= ci_upper:
    print("CI interpretation: The confidence interval includes 0, so the slope effect is not statistically clear.")
else:
    print("CI interpretation: The confidence interval excludes 0, so the slope effect is statistically clearer.")

print(f"Variance explained: yield curve slope explains about {r_squared:.4%} of next-month HSI return variation.")

Final dataset shape:
(197, 2)

First 5 rows:
            yield_curve_slope  HSI_next_month_return
2010-02-28               2.80               0.030601
2010-03-31               2.82              -0.006156
2010-04-30               2.72              -0.063642
2010-05-31               2.55               0.018406
2010-06-30               2.36               0.044752

Linear Regression Results
                              OLS Regression Results                             
Dep. Variable:     HSI_next_month_return   R-squared:                       0.000
Model:                               OLS   Adj. R-squared:                 -0.005
Method:                    Least Squares   F-statistic:                   0.03217
Date:                   Tue, 07 Jul 2026   Prob (F-statistic):              0.858
Time:                           15:57:28   Log-Likelihood:                 283.70
No. Observations:                    197   AIC:                            -563.4
Df Residuals:                       

A4: The US yield curve slope does not provide evidence of predictive power for next-month HSI returns in this test. The beta is insignificant, the confidence interval includes zero, and R² is essentially zero. This signal should be classified as likely noise, not tradeable.

Q5: Does it survive multiple-testing correction?


A5: No correction is strictly needed because only one hypothesis was tested. However, the signal is not significant even before correction, with p = 0.8578.

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error

# ------------------------------------------------------------
# Day 14: Normal Train/Test Split Regression
# ------------------------------------------------------------
# Question:
# Does the yield curve slope predict next-month HSI returns?
#
# Model:
# HSI_next_month_return = alpha + beta * yield_curve_slope + error
#
# Train/test split:
# Train on older data.
# Test on newer unseen data.
# ------------------------------------------------------------

tickers = {
    "HSI": "^HSI",
    "US_10Y": "^TNX",
    "US_short_rate": "^IRX",
}

data = yf.download(
    list(tickers.values()),
    period="20y",
    auto_adjust=True,
    progress=False
)

prices = data["Close"]

reverse_names = {symbol: name for name, symbol in tickers.items()}
prices = prices.rename(columns=reverse_names)

# ------------------------------------------------------------
# Convert daily data to monthly data
# ------------------------------------------------------------
# We use month-end values.
#
# monthly_prices["HSI"]:
# HSI price at the end of each month.
#
# monthly_prices["US_10Y"] and ["US_short_rate"]:
# Yield levels at the end of each month.
# ------------------------------------------------------------

monthly_prices = prices.resample("M").last()

# HSI monthly return:
# Percentage change in HSI from one month-end to the next.
monthly_hsi_return = monthly_prices["HSI"].pct_change(fill_method=None)

# Yield curve slope:
# Long-term yield minus short-term yield.
#
# If slope is high/positive:
# Long rates are above short rates.
#
# If slope is low/negative:
# Curve is flat or inverted.
yield_curve_slope = monthly_prices["US_10Y"] - monthly_prices["US_short_rate"]

# ------------------------------------------------------------
# Build prediction dataset
# ------------------------------------------------------------
# X = yield_curve_slope this month
# Y = HSI return next month
#
# shift(-1) means:
# move next month's HSI return into the current row.
#
# This lets us test whether today's slope predicts the future return.
# ------------------------------------------------------------

df = pd.DataFrame({
    "yield_curve_slope": yield_curve_slope,
    "HSI_next_month_return": monthly_hsi_return.shift(-1),
}).dropna()

print("Dataset shape:")
print(df.shape)

print("\nDate range:")
print(df.index.min(), "to", df.index.max())

# ------------------------------------------------------------
# Chronological train/test split
# ------------------------------------------------------------
# Train:
# Earlier data used to fit the model.
#
# Test:
# Later data the model has not seen.
#
# We do NOT randomly split financial time series because random splitting
# can mix future data into the training process.
# ------------------------------------------------------------

split_date = "2020-01-01"

train = df[df.index < split_date]
test = df[df.index >= split_date]

print("\nTrain/test sizes:")
print(f"Train observations: {len(train)}")
print(f"Test observations: {len(test)}")

# ------------------------------------------------------------
# Fit linear regression on training data
# ------------------------------------------------------------

Y_train = train["HSI_next_month_return"]
X_train = train[["yield_curve_slope"]]

# Add constant/intercept.
# This allows the model to estimate alpha.
X_train = sm.add_constant(X_train)

model = sm.OLS(Y_train, X_train).fit()

print("\nTRAINING REGRESSION RESULTS")
print(model.summary())

# ------------------------------------------------------------
# Predict on test data
# ------------------------------------------------------------

Y_test = test["HSI_next_month_return"]
X_test = test[["yield_curve_slope"]]
X_test = sm.add_constant(X_test)

test_predictions = model.predict(X_test)

# ------------------------------------------------------------
# Evaluate out-of-sample performance
# ------------------------------------------------------------
# Out-of-sample R²:
# How much test-period variation the model explains.
#
# It can be negative.
# Negative means the model is worse than simply predicting the average.
#
# RMSE:
# Root Mean Squared Error.
# Typical prediction error size.
# ------------------------------------------------------------

oos_r2 = r2_score(Y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(Y_test, test_predictions))

print("\nOUT-OF-SAMPLE RESULTS")
print(f"Out-of-sample R-squared: {oos_r2:.2%}")
print(f"RMSE: {rmse:.4%}")

# ------------------------------------------------------------
# Simple interpretation
# ------------------------------------------------------------

beta = model.params["yield_curve_slope"]
p_value = model.pvalues["yield_curve_slope"]

print("\nINTERPRETATION")
print(f"Training beta: {beta:.4%}")
print(f"Training p-value: {p_value:.4f}")

if p_value < 0.05:
    print("The yield curve slope is statistically significant in the training data.")
else:
    print("The yield curve slope is NOT statistically significant in the training data.")

if oos_r2 > 0:
    print("The model has positive out-of-sample explanatory power.")
else:
    print("The model has weak or negative out-of-sample explanatory power.")

Dataset shape:
(240, 2)

Date range:
2006-07-31 00:00:00 to 2026-06-30 00:00:00

Train/test sizes:
Train observations: 162
Test observations: 78

TRAINING REGRESSION RESULTS
                              OLS Regression Results                             
Dep. Variable:     HSI_next_month_return   R-squared:                       0.007
Model:                               OLS   Adj. R-squared:                  0.000
Method:                    Least Squares   F-statistic:                     1.060
Date:                   Wed, 08 Jul 2026   Prob (F-statistic):              0.305
Time:                           07:56:39   Log-Likelihood:                 223.96
No. Observations:                    162   AIC:                            -443.9
Df Residuals:                        160   BIC:                            -437.8
Df Model:                              1                                         
Covariance Type:               nonrobust                                         
      

/tmp/ipykernel_1785/1738962914.py:51: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_prices = prices.resample("M").last()


A6: The yield curve slope signal fails the out-of-sample test. In the training data, the beta was not statistically significant (p = 0.3048), and the estimated direction was negative rather than positive. In the test period, out-of-sample R² was -1.89%, meaning the model performed worse than a simple average-return baseline. Therefore, the signal does not show reliable predictive power for next-month HSI returns.

Q7: Is the effect large enough to trade?

A7: No. The effect is not large enough to trade because the signal is not statistically significant and fails out-of-sample validation. Since the relationship itself is unreliable, adding transaction costs would only make the result worse. Therefore, the signal should not be considered tradeable. Even if realistic trading costs were assumed, they would further reduce performance. But the main reason for rejection is more fundamental: the signal has no validated predictive edge.

Q8: Does it avoid obvious overfitting?

A8: The model avoids obvious overfitting because it uses a simple one-variable linear regression based on a pre-defined economic hypothesis. The train/test split is chronological, so future data is not used to train the model. There is no aggressive parameter tuning or complex model selection. However, avoiding overfitting does not make the signal useful: it still fails statistical significance and out-of-sample validation.